# LSTM으로 분류 모델 작성 (감성 분류)

In [13]:
docs = ['너무 재밌네요','최고에요','참 잘 만든 영화에요','추천하고 싶은 영화입니다','한 번 더 보고 싶네요',\
        '글쎄요','별로네요','생각보다 지루하네요','연기가 어색해요','재미없어요']
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer

labels = np.array([1, 1, 1, 1, 1, 0, 0, 0, 0, 0])
token = Tokenizer()
token.fit_on_texts(docs) # 문자열을 잘라서 단어 사전을 만듦
print(token.word_index)

x = token.texts_to_sequences(docs) # 토큰화
print(x)

# 입력 시퀀스 데이터를 딥러닝 모델에 넣기 전에 토큰의 길이를 동일하게 맞춰줘야 함. - 선행조건
from tensorflow.keras.utils import pad_sequences
padded_x = pad_sequences(x, maxlen=5, padding='pre') # post-뒤에 0을 채움
print('패딩 결과 :\n',padded_x)
#  [[ 0  0  0  1  2]
#  [ 0  0  0  0  3]

{'너무': 1, '재밌네요': 2, '최고에요': 3, '참': 4, '잘': 5, '만든': 6, '영화에요': 7, '추천하고': 8, '싶은': 9, '영화입니다': 10, '한': 11, '번': 12, '더': 13, '보고': 14, '싶네요': 15, '글쎄요': 16, '별로네요': 17, '생각보다': 18, '지루하네요': 19, '연기가': 20, '어색해요': 21, '재미없어요': 22}
[[1, 2], [3], [4, 5, 6, 7], [8, 9, 10], [11, 12, 13, 14, 15], [16], [17], [18, 19], [20, 21], [22]]
패딩 결과 :
 [[ 0  0  0  1  2]
 [ 0  0  0  0  3]
 [ 0  4  5  6  7]
 [ 0  0  8  9 10]
 [11 12 13 14 15]
 [ 0  0  0  0 16]
 [ 0  0  0  0 17]
 [ 0  0  0 18 19]
 [ 0  0  0 20 21]
 [ 0  0  0  0 22]]


In [14]:
# Deep Learning 모델 처리
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Embedding, Input, Flatten

word_size = len(token.word_index) + 1 # 가능한 토큰 갯수 + 1 - LSTM이 내부적으로 0을 붙임(패딩처리) 그래서 + 1을 해줘야함
model = Sequential()
model.add(Input(shape = (5, )))
model.add(Embedding(input_dim=word_size, output_dim=8)) # output_dim = 8 : 각 단어를 8차원 실수 벡터로 표현, 의미가 같은것끼리 같은 공간에 실수화됨
model.add(LSTM(32, activation='tanh'))
# model.add(Flatten()) # 출력이 이미 2차원 이므로 필요 없음
model.add(Dense(32, activation='relu'))
model.add(Dense(1, activation='sigmoid')) # 감성분류는 Sigmoid
print(model.summary())

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.fit(padded_x, labels, epochs=20, verbose=1)
print('정확도 :', model.evaluate(padded_x, labels)[1])
print('예측 :', np.where(model.predict(padded_x) > 0.5 , 1, 0).ravel())

Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_5 (Embedding)         │ (None, 5, 8)           │           184 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_5 (LSTM)                   │ (None, 32)             │         5,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 32)             │         1,056 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 6,521 (25.47 KB)

 Trainable params: 6,521 (25.47 KB)

 Non-trainable params: 0 (0.00 B)

None
Epoch 1/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step - accuracy: 0.5000 - loss: 0.6935
Epoch 2/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step - accuracy: 0.8000 - loss: 0.6923
Epoch 3/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step - accuracy: 0.8000 - loss: 0.6917
Epoch 4/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step - accuracy: 0.9000 - loss: 0.6912
Epoch 5/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step - accuracy: 1.0000 - loss: 0.6906
Epoch 6/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step - accuracy: 1.0000 - loss: 0.6903
Epoch 7/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step - accuracy: 1.0000 - loss: 0.6898
Epoch 8/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step - accuracy: 1.0000 - loss: 0.6893
Epoch 9/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step - accuracy: 0.9000 - loss: 0.6888
Epoch 10/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step - accuracy: 0.9000 - loss: 0.6882
Epoch 11/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step - accuracy: 0.9000 - loss: 0.6876
Epoch 12/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step - accuracy: 0.9000 - loss: 0.686